# Modelo Baseline. Pruebas varias

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

df = pd.read_csv("imdb_reviews.csv")

## Vectorización con TF-IFD

In [2]:
vectorizer = TfidfVectorizer(max_features=10000, 
                             ngram_range=(1, 2), 
                             stop_words="english")

X = vectorizer.fit_transform(df["review"]).toarray()
y = df["label"].copy()

# split train + temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# split dev y test
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_dev_scaled = scaler.transform(X_dev)
X_test_scaled = scaler.transform(X_test)

print(len(X_train_scaled), len(X_dev_scaled), len(X_test_scaled))

35000 7500 7500


## Redes densas con cantidad de neuronas decreciente. 
Intención: reducción de dimensionalidad

### Modelo Overkill. Máxima capacidad, regularización casi mínima
256 -> 128 -> 64 -> salida

In [3]:
# Definir el modelo de red neuronal
model = Sequential([
    # Capa de entrada y primera capa oculta con activación ReLU y regularización L2
    Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],), kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout para mitigar el sobreajuste

    # Segunda capa oculta con activación ReLU y regularización L2
    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Tercera capa oculta con activación ReLU y regularización L2
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Capa de salida con activación sigmoide para clasificación binaria
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Mostrar un resumen del modelo
print("\nResumen del modelo de red neuronal:")
model.summary()

c:\Github\inteligencia_artificial_tp\.venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Resumen del modelo de red neuronal:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │     2,560,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,601,473 (9.92 MB)

 Trainable params: 2,601,473 (9.92 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entrenar el modelo
print("\nEntrenando el modelo...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100, # Número máximo de épocas
    batch_size=32, # Tamaño del lote para el entrenamiento
    validation_data=(X_dev_scaled, y_dev), # Datos de validación
    callbacks=[early_stopping], # Aplicar Early Stopping
    verbose=1 # Mostrar progreso del entrenamiento
)

print("\nEntrenamiento completado.")


Entrenando el modelo...
Epoch 1/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 45s 37ms/step - accuracy: 0.8351 - loss: 1.2035 - val_accuracy: 0.8679 - val_loss: 1.0258
Epoch 2/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 44s 40ms/step - accuracy: 0.8972 - loss: 0.8508 - val_accuracy: 0.8659 - val_loss: 0.8373
Epoch 3/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 74s 32ms/step - accuracy: 0.9003 - loss: 0.7157 - val_accuracy: 0.8643 - val_loss: 0.7669
Epoch 4/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 35s 32ms/step - accuracy: 0.9127 - loss: 0.6611 - val_accuracy: 0.8627 - val_loss: 0.7561
Epoch 5/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 36s 33ms/step - accuracy: 0.9206 - loss: 0.6388 - val_accuracy: 0.8669 - val_loss: 0.7832
Epoch 6/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 36s 33ms/step - accuracy: 0.9260 - loss: 0.6413 - val_accuracy: 0.8631 - val_loss: 0.7978
Epoch 7/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 40s 32ms/step - accuracy: 0.9289 - loss: 0.6291 - val_accuracy: 0.8668 - val_loss: 0.8034
Epoch 8/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3

Ajusta bastante bien a los datos de train, pero casi no mejora el performance en el conjunto dev.

In [5]:
# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nPrecisión del modelo en el conjunto de prueba: {accuracy:.4f}")
print(f"Pérdida del modelo en el conjunto de prueba: {loss:.4f}")

# Realizar predicciones en el conjunto de prueba
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int) # Convertir probabilidades a clases binarias (0 o 1)

# Mostrar el reporte de clasificación
print("\nReporte de clasificación en el conjunto de prueba:")
print(classification_report(y_test, y_pred))


Precisión del modelo en el conjunto de prueba: 0.8641
Pérdida del modelo en el conjunto de prueba: 0.6224
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Reporte de clasificación en el conjunto de prueba:
              precision    recall  f1-score   support

           0       0.88      0.84      0.86      3750
           1       0.85      0.89      0.87      3750

    accuracy                           0.86      7500
   macro avg       0.86      0.86      0.86      7500
weighted avg       0.86      0.86      0.86      7500



Resultado aceptable/mediocre

### Red más pequeña
128 -> 64 -> 32 -> salida

In [6]:
# Definir el modelo de red neuronal
model = Sequential([
    # Capa de entrada y primera capa oculta con activación ReLU y regularización L2
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],), kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout para mitigar el sobreajuste

    # Segunda capa oculta con activación ReLU y regularización L2
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Tercera capa oculta con activación ReLU y regularización L2
    Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Capa de salida con activación sigmoide para clasificación binaria
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Mostrar un resumen del modelo
print("\nResumen del modelo de red neuronal:")
model.summary()


Resumen del modelo de red neuronal:


c:\Github\inteligencia_artificial_tp\.venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 128)            │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,290,497 (4.92 MB)

 Trainable params: 1,290,497 (4.92 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entrenar el modelo
print("\nEntrenando el modelo...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100, # Número máximo de épocas
    batch_size=32, # Tamaño del lote para el entrenamiento
    validation_data=(X_dev_scaled, y_dev), # Datos de validación
    callbacks=[early_stopping], # Aplicar Early Stopping
    verbose=1 # Mostrar progreso del entrenamiento
)

print("\nEntrenamiento completado.")


Entrenando el modelo...
Epoch 1/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 22s 18ms/step - accuracy: 0.8272 - loss: 0.9230 - val_accuracy: 0.8700 - val_loss: 0.8402
Epoch 2/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9041 - loss: 0.7235 - val_accuracy: 0.8648 - val_loss: 0.7435
Epoch 3/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9136 - loss: 0.6266 - val_accuracy: 0.8673 - val_loss: 0.7129
Epoch 4/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9189 - loss: 0.5872 - val_accuracy: 0.8645 - val_loss: 0.7082
Epoch 5/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9232 - loss: 0.5672 - val_accuracy: 0.8609 - val_loss: 0.7225
Epoch 6/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.9283 - loss: 0.5596 - val_accuracy: 0.8620 - val_loss: 0.7195
Epoch 7/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9321 - loss: 0.5491 - val_accuracy: 0.8633 - val_loss: 0.7140
Epoch 8/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 2

Ajuste relativamente mejor, pero aún con problemas de overfit.

In [8]:
# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nPrecisión del modelo en el conjunto de prueba: {accuracy:.4f}")
print(f"Pérdida del modelo en el conjunto de prueba: {loss:.4f}")

# Realizar predicciones en el conjunto de prueba
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int) # Convertir probabilidades a clases binarias (0 o 1)

# Mostrar el reporte de clasificación
print("\nReporte de clasificación en el conjunto de prueba:")
print(classification_report(y_test, y_pred))


Precisión del modelo en el conjunto de prueba: 0.8704
Pérdida del modelo en el conjunto de prueba: 0.7066
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Reporte de clasificación en el conjunto de prueba:
              precision    recall  f1-score   support

           0       0.87      0.87      0.87      3750
           1       0.87      0.87      0.87      3750

    accuracy                           0.87      7500
   macro avg       0.87      0.87      0.87      7500
weighted avg       0.87      0.87      0.87      7500



Mejora marginal en accuracy en test. 0.8641 -> 0.8704

### Misma red, penalización levemente más fuerte:
* l2 = 0.01

In [9]:
# Definir el modelo de red neuronal
model = Sequential([
    # Capa de entrada y primera capa oculta con activación ReLU y regularización L2
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],), kernel_regularizer=l2(0.01)),
    Dropout(0.3), # Capa Dropout para mitigar el sobreajuste

    # Segunda capa oculta con activación ReLU y regularización L2
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3), # Capa Dropout

    # Tercera capa oculta con activación ReLU y regularización L2
    Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3), # Capa Dropout

    # Capa de salida con activación sigmoide para clasificación binaria
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Mostrar un resumen del modelo
print("\nResumen del modelo de red neuronal:")
model.summary()


Resumen del modelo de red neuronal:


c:\Github\inteligencia_artificial_tp\.venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 128)            │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,290,497 (4.92 MB)

 Trainable params: 1,290,497 (4.92 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entrenar el modelo
print("\nEntrenando el modelo...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100, # Número máximo de épocas
    batch_size=32, # Tamaño del lote para el entrenamiento
    validation_data=(X_dev_scaled, y_dev), # Datos de validación
    callbacks=[early_stopping], # Aplicar Early Stopping
    verbose=1 # Mostrar progreso del entrenamiento
)

print("\nEntrenamiento completado.")


Entrenando el modelo...
Epoch 1/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - accuracy: 0.8277 - loss: 1.5044 - val_accuracy: 0.8473 - val_loss: 0.9378
Epoch 2/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - accuracy: 0.8513 - loss: 0.8965 - val_accuracy: 0.8475 - val_loss: 0.8893
Epoch 3/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.8572 - loss: 0.8603 - val_accuracy: 0.8544 - val_loss: 0.8248
Epoch 4/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - accuracy: 0.8607 - loss: 0.8100 - val_accuracy: 0.8573 - val_loss: 0.7663
Epoch 5/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - accuracy: 0.8611 - loss: 0.7522 - val_accuracy: 0.8539 - val_loss: 0.7137
Epoch 6/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - accuracy: 0.8659 - loss: 0.7061 - val_accuracy: 0.8520 - val_loss: 0.6853
Epoch 7/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 21s 16ms/step - accuracy: 0.8656 - loss: 0.6942 - val_accuracy: 0.8632 - val_loss: 0.6807
Epoch 8/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 1

In [11]:
# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nPrecisión del modelo en el conjunto de prueba: {accuracy:.4f}")
print(f"Pérdida del modelo en el conjunto de prueba: {loss:.4f}")

# Realizar predicciones en el conjunto de prueba
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int) # Convertir probabilidades a clases binarias (0 o 1)

# Mostrar el reporte de clasificación
print("\nReporte de clasificación en el conjunto de prueba:")
print(classification_report(y_test, y_pred))


Precisión del modelo en el conjunto de prueba: 0.8665
Pérdida del modelo en el conjunto de prueba: 0.6243
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Reporte de clasificación en el conjunto de prueba:
              precision    recall  f1-score   support

           0       0.86      0.87      0.87      3750
           1       0.87      0.86      0.87      3750

    accuracy                           0.87      7500
   macro avg       0.87      0.87      0.87      7500
weighted avg       0.87      0.87      0.87      7500



Se emparejan los valores de pérdida, más bajos en dev, y más altos en train. Sin mejora aparente

### Reducción de dimensionalidad con SVD

In [12]:
vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(df["review"]).toarray()
y = df["label"].copy()

# Reducción de dimensionalidad con SVD
svd = TruncatedSVD(n_components=300)
X_svd = svd.fit_transform(X)

# split train + temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X_svd, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# split dev y test
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_dev_scaled = scaler.transform(X_dev)
X_test_scaled = scaler.transform(X_test)

print(len(X_train_scaled), len(X_dev_scaled), len(X_test_scaled))

35000 7500 7500


In [13]:
# Definir el modelo de red neuronal
model = Sequential([
    # Capa de entrada y primera capa oculta con activación ReLU y regularización L2
    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],), kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout para mitigar el sobreajuste

    # Segunda capa oculta con activación ReLU y regularización L2
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Tercera capa oculta con activación ReLU y regularización L2
    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Capa de salida con activación sigmoide para clasificación binaria
    Dense(1, activation='sigmoid')
])


model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Mostrar un resumen del modelo
print("\nResumen del modelo de red neuronal:")
model.summary()

c:\Github\inteligencia_artificial_tp\.venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Resumen del modelo de red neuronal:


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 32)             │         9,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,193 (78.88 KB)

 Trainable params: 20,193 (78.88 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entrenar el modelo
print("\nEntrenando el modelo...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100, # Número máximo de épocas
    batch_size=32, # Tamaño del lote para el entrenamiento
    validation_data=(X_dev_scaled, y_dev), # Datos de validación
    callbacks=[early_stopping], # Aplicar Early Stopping
    verbose=1 # Mostrar progreso del entrenamiento
)

print("\nEntrenamiento completado.")


Entrenando el modelo...
Epoch 1/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7613 - loss: 0.5959 - val_accuracy: 0.8635 - val_loss: 0.4099
Epoch 2/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8635 - loss: 0.3997 - val_accuracy: 0.8712 - val_loss: 0.3679
Epoch 3/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8748 - loss: 0.3590 - val_accuracy: 0.8692 - val_loss: 0.3527
Epoch 4/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8786 - loss: 0.3407 - val_accuracy: 0.8659 - val_loss: 0.3562
Epoch 5/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8808 - loss: 0.3343 - val_accuracy: 0.8652 - val_loss: 0.3550
Epoch 6/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8831 - loss: 0.3300 - val_accuracy: 0.8657 - val_loss: 0.3542
Epoch 7/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8865 - loss: 0.3268 - val_accuracy: 0.8657 - val_loss: 0.3511
Epoch 8/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - a

In [15]:
# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nPrecisión del modelo en el conjunto de prueba: {accuracy:.4f}")
print(f"Pérdida del modelo en el conjunto de prueba: {loss:.4f}")

# Realizar predicciones en el conjunto de prueba
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int) # Convertir probabilidades a clases binarias (0 o 1)

# Mostrar el reporte de clasificación
print("\nReporte de clasificación en el conjunto de prueba:")
print(classification_report(y_test, y_pred))


Precisión del modelo en el conjunto de prueba: 0.8733
Pérdida del modelo en el conjunto de prueba: 0.3478
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

Reporte de clasificación en el conjunto de prueba:
              precision    recall  f1-score   support

           0       0.88      0.86      0.87      3750
           1       0.87      0.88      0.87      3750

    accuracy                           0.87      7500
   macro avg       0.87      0.87      0.87      7500
weighted avg       0.87      0.87      0.87      7500



Ampliamente superior en eficiencia y performance prácticamente idéntico en accuracy, y con mejoras en los valores de pérdida.

Aún así, los resultados de este notebook probablemente esten siendo sumamente afectados porlas limitaciones de las features, producto de una vectorización básica y un NLP pobre.